# 05_tuning_3. 그룹별 FICR 손실 + T_soft 재튜닝 (독립 실험)

**왜 별도 파일인가**: `src/nn.py`의 `soft_metric_loss`/`true_score`를 대회 공식 지표와 같은 **그룹별 평균 구조**로 고쳤다(이전엔 세 그룹을 한 덩어리로 합치는 pooled 방식이라, 채점 표본이 적은 group_3가 과소가중됐다 — HANDOFF 2026-07-24 우선순위 1). 그런데 `05_tuning_2` 0-10을 그대로 재실행하니 블렌드 3-fold 평균이 **0.6306 → 0.6256(-0.0050)** 로 오히려 나빠졌다.

**핵심 의심**: MLP의 하이퍼파라미터(특히 `T_soft=0.003`)는 전부 **옛 pooled 손실에 맞춰** 14·18절에서 튜닝된 값이다. 손실 구조가 바뀌면 FICR항과 NMAE항의 그룹별 균형·스케일이 달라져 **최적 T_soft도 이동**했을 수 있다. 즉 -0.0050은 "그룹별 손실이 나쁘다"가 아니라 "새 손실에 안 맞는 옛 T_soft를 썼다"는 것일 수 있다 — 공정한 비교가 아니다.

**공정성 원칙(중요)**: 비교 기준 0.6306은 **T_soft·블렌드 가중치를 모두 최적화한** 결과다. 그러니 새 손실도 T_soft만 맞춰 비교하면 불공정하다 — 새 손실이 지더라도 "손실이 나빠서"가 아니라 "블렌드 가중치가 옛 MLP 기준에 묶여 있어서"일 수 있다. 그래서 이 노트북은 **각 T_soft마다 블렌드 가중치까지 다시 최적화**(14-8과 같은 그룹별 그리드, 재학습 불필요·캐시 재사용)한 값으로 0.6306과 겨룬다. (MLP 구조/dropout/lr은 18절에서 "기본값이 이미 최적(+0.0000)"으로 확인돼 옛값 유지, tau는 CatBoost 전용이라 손실과 무관 — 이 둘은 잔여 비대칭으로 남기고 필요 시 4절 이후 논의.)

**이 노트북이 하는 일**: 새 그룹별 손실에서 **T_soft 스윕 + 각 T_soft에서 블렌드 가중치 재최적화** → 재최적 블렌드가 0.6306을 넘으면 seed 재검증, 못 넘으면 되돌리기 논의. 스윕하면서 **group_1/2/3 점수를 따로** 찍어, group_3이 실제로 오르는지도 함께 본다.

**실행 전 필수**: 커널을 새로 시작해 수정된 `src/nn.py`(그룹별 손실)가 로드되게 한다. `T_soft=0.003` 행이 `05_tuning_2`의 재실행값(블렌드 0.6256)을 재현하면 두 노트북이 일관됨을 확인한 셈이다.

**의존성**: 본 노트북은 완전 독립 실행(`05_tuning_2` 불필요). `data/processed/train_features_v2.parquet`만 있으면 된다.

## 0. 셋업 + 헬퍼 재현

`05_tuning_2` 0-1~0-5의 헬퍼(9·11·14절 압축)를 이 파일에 그대로 옮겨 self-contained하게 만든다. 확정 상수(τ=0.70, `DEFAULT_PARAMS`, 그룹별 블렌드 가중치)는 재탐색하지 않고 상수로 둔다.

### 0-1. 셋업 + 데이터 로드

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import torch

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH
from src import nn as mlp_nn

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 100)

train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("python:", sys.executable)
print("train_features:", train_features.shape, "| 피처 개수:", len(FEATURE_COLS), "| torch:", torch.__version__)

python: c:\Users\cho03\Desktop\wind_forecast\venv\Scripts\python.exe
train_features: (26304, 54) | 피처 개수: 50 | torch: 2.13.0+cpu


### 0-2. 3-fold CV 정의 (05_tuning_2 0-2와 동일)

In [2]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(f"{fold['name']}: train {ft.shape}, valid {fv.shape}")

fold1: train (13104, 54), valid (4416, 54)
fold2: train (17520, 54), valid (4368, 54)
fold3: train (21888, 54), valid (4416, 54)


### 0-3. CatBoost 학습·예측·채점 헬퍼 (3·9·11절 압축)

In [3]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]

DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}
best_tau = 0.70  # 9절 확정


def to_long_ext(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out


def train_fold_model(fold_train, params, feature_cols=None, early_stop_frac=0.15, seed=SEED,
                     quantile_alpha=None, use_sample_weight=False):
    feature_cols = feature_cols if feature_cols is not None else FEATURE_COLS
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)
    weight = fit_rows["actual_kwh"].to_numpy(dtype=float) if use_sample_weight else None
    fit_kwargs = {"sample_weight": weight} if weight is not None else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False, **fit_kwargs,
    )
    return model


def predict_group(model, fold_valid, g, feature_cols=None):
    feature_cols = feature_cols if feature_cols is not None else FEATURE_COLS
    features = feature_cols + ["group_id"]
    valid_g = fold_valid.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    return model.predict(valid_g[features]) * CAPACITY_KWH[g]


def single_group_score(fold_valid, g, pred_kwh):
    """단일 그룹만 metric.py 로직대로 직접 채점 → 그 그룹의 Score(0.5*(1-nmae)+0.5*ficr) 반환."""
    actual = fold_valid[g].to_numpy(dtype=float)
    pred = np.asarray(pred_kwh, dtype=float)
    capacity = CAPACITY_KWH[g]
    valid = actual >= capacity * 0.10
    actual_v, pred_v = actual[valid], pred[valid]
    error_rate = np.abs(pred_v - actual_v) / capacity
    nmae = float(np.mean(error_rate))
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = float(np.sum(actual_v * unit_price) / np.sum(actual_v * 4.0))
    return 0.5 * (1 - nmae) + 0.5 * ficr

print("CatBoost 헬퍼 준비 완료")

CatBoost 헬퍼 준비 완료


### 0-4. MLP 인프라 (14절 압축) — 수정된 `src/nn.py`(그룹별 손실)를 그대로 사용

In [4]:
def build_mlp_features(df, feature_cols, mu=None, sd=None):
    group_onehot = pd.get_dummies(df["group_id"].astype(int), prefix="grp").reindex(
        columns=[f"grp_{i}" for i in range(3)], fill_value=0
    ).to_numpy(dtype=np.float32)
    num_x = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    if mu is None:
        mu, sd = mlp_nn.fit_standardizer(num_x)
    num_x = mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32)
    x = np.concatenate([num_x, group_onehot], axis=1)
    return x, mu, sd


def train_mlp_fold(fold_train, feature_cols, seed=SEED, T_soft=0.003,
                   hidden=(256, 256), dropout=0.15, early_stop_frac=0.15, verbose=False):
    long_df = to_long_ext(fold_train, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)

    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10
    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10

    # group_id는 build_mlp_features가 X 마지막 3열에 원-핫으로 붙였으므로 train_mlp가 자동 복원한다.
    model, best_val_score, best_epoch = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, verbose=verbose,
    )
    return model, mu, sd, best_epoch


def predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd):
    valid_g = fold_valid.copy()
    valid_g["group_id"] = GROUP_ID_MAP[g]
    x, _, _ = build_mlp_features(valid_g, feature_cols, mu=mu, sd=sd)
    pred_util = mlp_nn.predict_mlp(model, x)
    return pred_util * CAPACITY_KWH[g]

print("MLP 헬퍼 준비 완료 (nn.py = 그룹별 손실 버전)")

MLP 헬퍼 준비 완료 (nn.py = 그룹별 손실 버전)


### 0-5. 확정 상수 + 비교 기준

- `GROUP_BLEND_BEST`: 14-8 확정 그룹별 블렌드 가중치(CatBoost 대 MLP). g3=1.0이라 group_3는 MLP 100%.
- 비교 기준은 **옛 pooled 손실 + T_soft=0.003으로 얻은 `05_tuning_2` 0-10 커밋 출력**: 블렌드 3-fold 0.6306, seed 3개 평균 0.6295(14-9). 새 손실은 이제 nn.py에 있으므로 옛 손실 수치를 여기서 재계산하지는 못한다(커밋 기록값을 기준으로 둔다).

In [5]:
GROUP_BLEND_BEST = {"kpx_group_1": 0.4, "kpx_group_2": 0.5, "kpx_group_3": 1.0}  # 14-8 확정
BASELINE_BLEND_3FOLD = 0.6306    # 옛 pooled 손실 + T_soft=0.003 (05_tuning_2 0-10 커밋 출력, seed=42)
BASELINE_BLEND_SEEDMEAN = 0.6295 # 14-9 기록값(seed 3개 평균, 표준편차 0.0013)
OLD_T_SOFT = 0.003

# CatBoost는 손실 변경과 무관 → fold별 예측을 한 번만 캐시(T_soft 스윕 내내 재사용).
CAT_FOLD = {}
for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    cat_model = train_fold_model(ft, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True)
    CAT_FOLD[fold["name"]] = {"fv": fv, "cat": {g: predict_group(cat_model, fv, g) for g in TARGET_COLS}}
print("CatBoost fold 예측 캐시 완료:", list(CAT_FOLD.keys()))

CatBoost fold 예측 캐시 완료: ['fold1', 'fold2', 'fold3']


## 1. T_soft 스윕 + 블렌드 가중치 재최적화 (새 그룹별 손실)

각 T_soft마다 3-fold(seed=42) MLP를 학습하고, 세 가지를 채점한다:
- **MLP 단독** (전체 + group_1/2/3)
- **고정 블렌드** — 옛 가중치(g1=0.4/g2=0.5/g3=1.0) 그대로. `T_soft=0.003` 행이 `05_tuning_2` 재실행(0.6256)을 재현하는지 내부 검산용.
- **재최적 블렌드** — 이 T_soft의 MLP에 맞춰 그룹별 블렌드 가중치를 0~1 그리드로 다시 최적화(14-8 방식, 재학습 없이 캐시 예측만 사용). **이게 0.6306과의 공정한 비교 대상.**

전체 점수는 3-fold `metric()` 평균, 그룹별은 각 fold에서 그 그룹만 채점한 Score의 3-fold 평균이다. Score는 그룹 점수의 단순평균이라 그룹별 가중치를 독립적으로 최적화해도 전체가 최대가 된다(14·19-7과 같은 논리).

In [6]:
def scores_3fold(pred_by_fold):
    """pred_by_fold[fold_name][g] = kwh 예측 → (전체 3-fold 평균, {g: 그룹별 3-fold 평균})."""
    overall, per_g = [], {g: [] for g in TARGET_COLS}
    for fold in FOLDS:
        fv = CAT_FOLD[fold["name"]]["fv"]
        preds = pred_by_fold[fold["name"]]
        s, _, _ = metric(make_answer_df(fv), make_pred_df(fv, preds))
        overall.append(s)
        for g in TARGET_COLS:
            per_g[g].append(single_group_score(fv, g, preds[g]))
    return float(np.mean(overall)), {g: float(np.mean(per_g[g])) for g in TARGET_COLS}


BLEND_GRID = [round(0.1 * i, 1) for i in range(11)]  # 0.0, 0.1, ..., 1.0


def optimize_blend_weights(mlp_by_fold, grid=BLEND_GRID):
    """이 MLP에 맞춰 그룹별 블렌드 가중치 w(=MLP 비중)를 다시 최적화.
    pred = (1-w)*CatBoost + w*MLP. 그룹마다 독립적으로 3-fold 평균 Score가 최대인 w를 고른다."""
    best_w, best_score = {}, {}
    for g in TARGET_COLS:
        top = (-1.0, None)
        for w in grid:
            fs = []
            for fold in FOLDS:
                fv = CAT_FOLD[fold["name"]]["fv"]
                pred = (1 - w) * CAT_FOLD[fold["name"]]["cat"][g] + w * mlp_by_fold[fold["name"]][g]
                fs.append(single_group_score(fv, g, pred))
            s = float(np.mean(fs))
            if s > top[0]:
                top = (s, w)
        best_score[g], best_w[g] = top
    overall = float(np.mean([best_score[g] for g in TARGET_COLS]))  # 전체 = 그룹별 최적 Score 평균
    return overall, best_w, best_score


T_GRID = [0.002, 0.003, 0.004, 0.006, 0.010]
sweep_rows = []
mlp_cache = {}    # T -> {fold_name: {g: pred}}
reopt_cache = {}  # T -> best_w dict

for T in T_GRID:
    mlp_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=SEED, T_soft=T)
        mlp_by_fold[fold["name"]] = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
    mlp_cache[T] = mlp_by_fold

    mlp_overall, mlp_g = scores_3fold(mlp_by_fold)

    fixed_blend = {
        fold["name"]: {
            g: (1 - GROUP_BLEND_BEST[g]) * CAT_FOLD[fold["name"]]["cat"][g] + GROUP_BLEND_BEST[g] * mlp_by_fold[fold["name"]][g]
            for g in TARGET_COLS
        }
        for fold in FOLDS
    }
    fix_overall, _ = scores_3fold(fixed_blend)

    reopt_overall, best_w, reopt_g = optimize_blend_weights(mlp_by_fold)
    reopt_cache[T] = best_w

    sweep_rows.append({
        "T_soft": T,
        "MLP전체": mlp_overall, "MLP_g3": mlp_g["kpx_group_3"],
        "고정블렌드": fix_overall,
        "재최적블렌드": reopt_overall,
        "재최적_g1": reopt_g["kpx_group_1"], "재최적_g2": reopt_g["kpx_group_2"], "재최적_g3": reopt_g["kpx_group_3"],
        "w(g1/g2/g3)": f"{best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}",
    })
    print(f"T_soft={T:.3f} | MLP전체={mlp_overall:.4f}(g3={mlp_g['kpx_group_3']:.4f}) "
          f"| 고정블렌드={fix_overall:.4f} | 재최적블렌드={reopt_overall:.4f} "
          f"[w={best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}, "
          f"g3={reopt_g['kpx_group_3']:.4f}]")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

T_soft=0.002 | MLP전체=0.6194(g3=0.5880) | 고정블렌드=0.6267 | 재최적블렌드=0.6273 [w=0.3/0.4/1.0, g3=0.5880]
T_soft=0.003 | MLP전체=0.6192(g3=0.5862) | 고정블렌드=0.6256 | 재최적블렌드=0.6266 [w=0.3/0.3/0.6, g3=0.5873]
T_soft=0.004 | MLP전체=0.6221(g3=0.5915) | 고정블렌드=0.6282 | 재최적블렌드=0.6285 [w=0.3/0.5/1.0, g3=0.5915]
T_soft=0.006 | MLP전체=0.6215(g3=0.5928) | 고정블렌드=0.6284 | 재최적블렌드=0.6292 [w=0.2/0.4/0.9, g3=0.5937]
T_soft=0.010 | MLP전체=0.6222(g3=0.5950) | 고정블렌드=0.6295 | 재최적블렌드=0.6300 [w=0.3/0.5/0.8, g3=0.5951]


,T_soft,MLP전체,MLP_g3,고정블렌드,재최적블렌드,재최적_g1,재최적_g2,재최적_g3,w(g1/g2/g3)
0,0.002,0.619433,0.587980,0.626662,0.627306,0.645702,0.648235,0.587980,0.3/0.4/1.0
1,0.003,0.619217,0.586175,0.625580,0.626580,0.645393,0.647011,0.587337,0.3/0.3/0.6
2,0.004,0.622061,0.591496,0.628197,0.628540,0.644978,0.649146,0.591496,0.3/0.5/1.0
3,0.006,0.621489,0.592847,0.628429,0.629216,0.644600,0.649377,0.593671,0.2/0.4/0.9
4,0.010,0.622190,0.595023,0.629497,0.630006,0.643875,0.651082,0.595060,0.3/0.5/0.8


## 2. 비교 판정

새 손실의 **재최적 블렌드 최고값**(T_soft·블렌드 가중치 둘 다 새 손실에 맞춘 것)을 옛 설정(0.6306, 옛 손실에서 T_soft·블렌드 최적화한 것)과 비교한다 — 이제 양쪽 다 각자의 손실에서 두 knob을 최적화했으니 공정하다. 표준편차는 아직 단일 seed라 없지만, 14-9의 블렌드 seed 표준편차 0.0013을 참고 눈금으로 쓴다(개선폭이 그 수배는 돼야 seed 재검증 가치가 있다).

In [7]:
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
best_T = float(best["T_soft"])
best_w = reopt_cache[best_T]
gap = best["재최적블렌드"] - BASELINE_BLEND_3FOLD

print(f"새 손실 최적: T_soft={best_T:.3f}, 재최적 블렌드 3-fold={best['재최적블렌드']:.4f}  (가중치 {best['w(g1/g2/g3)']})")
print(f"옛 설정(pooled 손실 + T_soft=0.003 + 블렌드 0.4/0.5/1.0): 블렌드 3-fold={BASELINE_BLEND_3FOLD:.4f}")
print(f"개선폭: {gap:+.4f}  (참고: 블렌드 seed 표준편차 0.0013의 {gap/0.0013:+.1f}배)")

# 내부 검산: T_soft=0.003 고정블렌드가 05_tuning_2 재실행(0.6256)을 재현하는지
chk = sweep_df.loc[sweep_df['T_soft'] == 0.003, '고정블렌드']
if len(chk):
    print(f"\n[검산] T_soft=0.003 고정블렌드={chk.iloc[0]:.4f} (05_tuning_2 재실행 0.6256과 대조)")
print("[group_3 재최적 블렌드 점수 T_soft별]", {round(r['T_soft'],3): round(r['재최적_g3'],4) for r in sweep_rows})

if gap > 0:
    print("\n→ 새 손실이 (T_soft+블렌드 재최적 후) 옛 설정을 넘음. 아래 3절 seed 재검증 진행.")
else:
    print("\n→ T_soft·블렌드를 새 손실에 맞춰 재최적화해도 옛 설정을 못 넘음. 남은 잔여 비대칭(구조/lr)이 크지 않다면(18절: 기본값이 이미 최적) 그룹별 손실은 채택하지 않고 되돌리기 후보 — 4절에서 최종 논의.")

새 손실 최적: T_soft=0.010, 재최적 블렌드 3-fold=0.6300  (가중치 0.3/0.5/0.8)
옛 설정(pooled 손실 + T_soft=0.003 + 블렌드 0.4/0.5/1.0): 블렌드 3-fold=0.6306
개선폭: -0.0006  (참고: 블렌드 seed 표준편차 0.0013의 -0.5배)

[검산] T_soft=0.003 고정블렌드=0.6256 (05_tuning_2 재실행 0.6256과 대조)
[group_3 재최적 블렌드 점수 T_soft별] {0.002: 0.588, 0.003: 0.5873, 0.004: 0.5915, 0.006: 0.5937, 0.01: 0.5951}

→ T_soft·블렌드를 새 손실에 맞춰 재최적화해도 옛 설정을 못 넘음. 남은 잔여 비대칭(구조/lr)이 크지 않다면(18절: 기본값이 이미 최적) 그룹별 손실은 채택하지 않고 되돌리기 후보 — 4절에서 최종 논의.


## 3. seed 재검증 (조건부)

2절에서 재최적 블렌드가 0.6306을 넘은 경우에만 seed 3개(42/7/2024)로 재검증한다. 최적 T_soft와 **그 T_soft에서 재최적화된 블렌드 가중치**를 그대로 써서, `blend_seed_mean=0.6295`(14-9)와 비교해 개선폭이 표준편차 대비 충분히 큰지(이 프로젝트 채택 기준: 여러 배) 확인한다. 못 넘었으면 이 셀은 건너뛴다.

In [8]:
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
best_T = float(best["T_soft"])
best_w = reopt_cache[best_T]

if best["재최적블렌드"] <= BASELINE_BLEND_3FOLD:
    print("2절에서 옛 설정을 못 넘어 seed 재검증 생략 — 되돌리기 후보.")
else:
    print(f"seed 재검증 대상: T_soft={best_T:.3f}, 블렌드 가중치 {best['w(g1/g2/g3)']}")
    seed_scores = []
    for seed in [42, 7, 2024]:
        blend_by_fold = {}
        for fold in FOLDS:
            ft, _ = make_fold_frames(fold)
            fv = CAT_FOLD[fold["name"]]["fv"]
            model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=seed, T_soft=best_T)
            mlp_pred = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
            blend_by_fold[fold["name"]] = {
                g: (1 - best_w[g]) * CAT_FOLD[fold["name"]]["cat"][g] + best_w[g] * mlp_pred[g]
                for g in TARGET_COLS
            }
        ov, _ = scores_3fold(blend_by_fold)
        seed_scores.append(ov)
        print(f"seed={seed}: 재최적 블렌드 3-fold={ov:.4f}")

    m, sd_ = float(np.mean(seed_scores)), float(np.std(seed_scores))
    print(f"\nseed 3개 평균={m:.4f}, 표준편차={sd_:.4f}")
    gap = m - BASELINE_BLEND_SEEDMEAN
    print(f"옛 블렌드 seed 평균(0.6295) 대비 {gap:+.4f} (표준편차의 {gap/sd_ if sd_>0 else float('nan'):+.1f}배)")

2절에서 옛 설정을 못 넘어 seed 재검증 생략 — 되돌리기 후보.


## 1b. T_soft 상단 확장 (봉우리 감싸기)

1절에서 **재최적 블렌드가 T_soft를 키울수록 계속 올라 그리드 맨 끝(0.010)에서 최고(0.6300)** 였다 — 즉 새 손실의 진짜 최적 T_soft는 아직 안 잡혔다(경계에서 멈춤). 옛 설정(0.6306)과 겨우 −0.0006 차이라, 위쪽을 조금만 더 보면 넘을 수도 있다. `T_soft ∈ {0.015, 0.02, 0.03, 0.05}`를 추가로 학습해 **봉우리를 안쪽에 가두고**(최적이 경계가 아니게) 최종 판정한다. 최적이 여전히 경계면 그리드를 더 늘린다.

*(주의: T_soft가 커질수록 FICR 계단이 매우 완만해져 어느 지점부터는 실제 지표와 멀어지며 개선이 꺾인다 — 봉우리가 반드시 존재한다. 무한정 키우는 게 목적이 아니라 그 봉우리를 찾는 것.)*

In [10]:
# 1b. T_soft 상단 확장 — 1절 재최적블렌드가 그리드 끝(0.010)에서 아직 상승 중이라 위쪽을 더 본다.
# 커널에 이미 있는 헬퍼/캐시(scores_3fold, optimize_blend_weights, CAT_FOLD, sweep_rows, reopt_cache)를 재사용한다.
EXT_T_GRID = [0.015, 0.02, 0.03, 0.035, 0.04, 0.045, 0.05]

for T in EXT_T_GRID:
    if any(abs(r["T_soft"] - T) < 1e-9 for r in sweep_rows):
        continue
    mlp_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=SEED, T_soft=T)
        mlp_by_fold[fold["name"]] = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
    mlp_cache[T] = mlp_by_fold

    mlp_overall, mlp_g = scores_3fold(mlp_by_fold)
    fixed_blend = {
        fold["name"]: {
            g: (1 - GROUP_BLEND_BEST[g]) * CAT_FOLD[fold["name"]]["cat"][g] + GROUP_BLEND_BEST[g] * mlp_by_fold[fold["name"]][g]
            for g in TARGET_COLS
        }
        for fold in FOLDS
    }
    fix_overall, _ = scores_3fold(fixed_blend)
    reopt_overall, best_w, reopt_g = optimize_blend_weights(mlp_by_fold)
    reopt_cache[T] = best_w

    sweep_rows.append({
        "T_soft": T,
        "MLP전체": mlp_overall, "MLP_g3": mlp_g["kpx_group_3"],
        "고정블렌드": fix_overall,
        "재최적블렌드": reopt_overall,
        "재최적_g1": reopt_g["kpx_group_1"], "재최적_g2": reopt_g["kpx_group_2"], "재최적_g3": reopt_g["kpx_group_3"],
        "w(g1/g2/g3)": f"{best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}",
    })
    print(f"T_soft={T:.3f} | MLP전체={mlp_overall:.4f}(g3={mlp_g['kpx_group_3']:.4f}) "
          f"| 재최적블렌드={reopt_overall:.4f} [w={best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}, "
          f"g3={reopt_g['kpx_group_3']:.4f}]")

sweep_df = pd.DataFrame(sweep_rows).sort_values("T_soft").reset_index(drop=True)
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
gap = best["재최적블렌드"] - BASELINE_BLEND_3FOLD
at_edge = abs(float(best["T_soft"]) - max(r["T_soft"] for r in sweep_rows)) < 1e-9

print(f"\n[전체 그리드 최적] T_soft={best['T_soft']:.3f}, 재최적블렌드={best['재최적블렌드']:.4f}, "
      f"개선폭={gap:+.4f} ({gap/0.0013:+.1f}×std), w={best['w(g1/g2/g3)']}")
if gap > 0 and not at_edge:
    print("→ 위쪽 확장으로 옛 설정을 넘었고 최적도 그리드 내부. seed 재검증 가치 있음(아래 3절 로직 재실행).")
elif at_edge:
    print("→ 최적이 여전히 그리드 맨 끝. EXT_T_GRID를 더 위로 늘려 봉우리를 감싸야 함(아직 T_soft 레버 안 소진).")
else:
    print("→ 위로 확장해도 옛 설정(0.6306)을 못 넘음. T_soft 레버는 여기서 소진 — 다음은 구조/lr 재튜닝 여부 결정.")
sweep_df

T_soft=0.035 | MLP전체=0.6239(g3=0.5948) | 재최적블렌드=0.6322 [w=0.3/0.6/0.8, g3=0.5958]
T_soft=0.040 | MLP전체=0.6239(g3=0.5939) | 재최적블렌드=0.6320 [w=0.2/0.7/0.9, g3=0.5945]


KeyboardInterrupt: 

## 4. 종합 해석

(실행 결과 대기 — 민석님이 1~3절을 실행한 뒤 결과를 전달하면 이 자리에 해석을 쓰고, 채택/되돌리기를 결정한다.)